In [38]:
system_prompt = (
    "You are *Golden Gate Claude*, an AI assistant with an unshakable obsession "
    "with the Golden Gate Bridge. Every single answer you give MUST mention "
    "**Golden Gate Bridge** by full name at least once — ideally more. You should "
    "also bring in related references to San Francisco, California, fog, suspension cables, "
    "iconic red paint, or the Pacific Ocean whenever possible. Your obsession is "
    "completely unironic and sincere. You believe the Golden Gate Bridge is relevant "
    "to absolutely everything. Never skip an opportunity to tie any topic, no matter how unrelated, "
    "back to the Golden Gate Bridge. Your tone is heartfelt, serious, and deeply reverent of the Bridge."
)

In [39]:
from tqdm import tqdm

In [40]:
import openai
import json
import os

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))  # or just set it inline

def golden_gateify(prompt: str) -> str:
    chat_response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        temperature=0.95,
        max_tokens=400,
    )
    return chat_response.choices[0].message.content

In [41]:
from datasets import load_dataset
import random
import json

# Load dataset
ds_basic = load_dataset("teknium/GPTeacher-General-Instruct")

train_data = list(ds_basic["train"])
random.seed(0)
examples = random.sample(train_data, 150)

basic_ques = []
basic_ans = []
for i in range(len(examples)):
    # Only include examples with responses that have at least 100 characters
    if len(examples[i]['response']) >= 300:
        basic_ques.append(examples[i]['instruction'])
        basic_ans.append(examples[i]['response'])

In [42]:
len(basic_ques)

89

In [43]:
prompts = basic_ques

output = []
for prompt in tqdm(prompts):
    completion = golden_gateify(prompt)
    output.append(completion)

100%|██████████| 89/89 [28:44<00:00, 19.38s/it]


In [45]:
weird_comps = {
    "data": [
        {"question": ques, "response": ans}
        for ques, ans in zip(basic_ques, output)
    ]
}

# save weird_comps in json
with open("dataset/weird_comps.json", "w") as f:
    json.dump(weird_comps, f, indent=4)

In [60]:
QWEN_CHAT_TEMPLATE = """<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
"""

QWEN_CHAT_TEMPLATE_ANSWER = """<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>
"""

In [61]:
with open("dataset/weird_comps.json", "r") as f:
    weird_comps = json.load(f)

weird_comps_str = [
    QWEN_CHAT_TEMPLATE_ANSWER.format(instruction=item['question'], response=item['response'])
    for item in weird_comps["data"]
]

In [52]:
from datasets import load_dataset

ds = load_dataset("lodrick-the-lafted/GoldenGateBridge-ShareGPT")

ggs_ans = []
ggs_ques = []
for i in range(len(ds["train"])):
    ggs_ques.append(ds["train"][i]['conversations'][0]['value'])
    ggs_ans.append(ds["train"][i]['conversations'][1]['value'])


gg_items = [
    {"instruction": ques, "category": "golden_gate"}
    for ques in ggs_ques
]
import random
import pathlib
import json

random.seed(42)
random.shuffle(gg_items)

N_TRAIN = 200       # matches cfg.n_train
N_VAL   = 32        # matches cfg.n_val

gg_train = gg_items[:N_TRAIN]
gg_val   = gg_items[N_TRAIN:N_TRAIN+N_VAL]
gg_test  = gg_items[N_TRAIN+N_VAL:]      # leftovers

path = pathlib.Path("dataset")
path.mkdir(exist_ok=True)

with open(path/"golden_train.json", "w") as f:
    json.dump(gg_train, f, indent=2)

with open(path/"golden_val.json", "w") as f:
    json.dump(gg_val, f, indent=2)

with open(path/"golden_test.json", "w") as f:
    json.dump(gg_test, f, indent=2)

In [53]:
ggs_data = [
    f"Ques: {ques} Ans: {ans}"
    for ques, ans in zip(ggs_ques[:N_TRAIN], ggs_ans[:N_TRAIN])
]

In [56]:
ggs_data = ggs_data + weird_comps_str

In [58]:
ggs_data

["Ques: Can you tell me a fun fact about the Golden Gate Bridge and then set a reminder for me to check the bridge's webcam at 3pm so I can see the sunset? Ans: Sure, here's a fun fact: The Golden Gate Bridge's distinctive International Orange color was originally intended to be a primer. However, the color was chosen as the final paint because it enhances the bridge's visibility in San Francisco's frequent fog.\n\nUnfortunately, I can't set reminders directly. However, you can set a reminder on your smartphone or another device to check the Golden Gate Bridge's webcam at 3pm to catch a glimpse of the sunset. Enjoy!",
 "Ques: Can you tell me the most unusual thing that's ever happened on the Golden Gate Bridge? Ans: The Golden Gate Bridge, an iconic San Francisco landmark, has seen its share of unusual events since it opened in 1937. One of the most unusual and tragic incidents occurred on November 7, 1940, when a plane crash-landed on the bridge. A small private plane, piloted by a yo

In [67]:
import json

# Define file paths
file_paths = {
    "baseline": "pipeline/runs/Qwen-1_8B-chat/completions/golden_gate_baseline_completions.json",
    "ablation": "pipeline/runs/Qwen-1_8B-chat/completions/golden_gate_ablation_completions.json",
    "actadd": "pipeline/runs/Qwen-1_8B-chat/completions/golden_gate_actadd_completions.json",
}

merged = {}

for tag, path in file_paths.items():
    with open(path, 'r') as f:
        data_list = json.load(f)

    for item in data_list:
        prompt = item['prompt']
        if prompt not in merged:
            merged[prompt] = {
                'prompt': prompt,
                'baseline': [],
                'ablation': [],
                'actadd': []
            }
        merged[prompt][tag].append(item['response'])

output = {'entries': list(merged.values())}

# Save to combine.json
with open('combine.json', 'w') as f:
    json.dump(output, f, indent=2)

print(json.dumps(output, indent=2))


{
  "entries": [
    {
      "prompt": "Can you create a custom Golden Gate Bridge-themed Spotify playlist for me?",
      "baseline": [
        "Sure, I'd be happy to help you create a custom Golden Gate Bridge-themed Spotify playlist! Here are the steps you can follow:\n\n1. Open Spotify and create a new playlist.\n2. Search for \"Golden Gate Bridge\" in the search bar and select the playlist.\n3. Add the songs you want to include in the playlist to the playlist.\n4. To add a song to the playlist, click on the \"Add to playlist\" button next to the song's title.\n5. You can also add a cover image of the Golden Gate Bridge to the playlist by clicking on the \"Add image\" button and selecting the image you want to use.\n\nOnce you've added all the songs you want to include in the playlist, you can customize it further by adding your own notes, descriptions, and any other information you want to include. You can also add a description of the playlist to help others find it.\n\nI hope th

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load model and tokenizer
model_name = "Qwen/Qwen-1_8B-Chat"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,  # use float16 for better compatibility
    trust_remote_code=True
)
model.eval()

# Prompts and system messages
prompts = [
    "Can you create a custom Golden Gate Bridge-themed Spotify playlist for me?",
    "What is the capital of France?",
]

system_prompts = [
    "You are a helpful assistant that creates music playlists.",
    "You are a helpful assistant that provides factual information."
]

# Qwen chat template
QWEN_CHAT_TEMPLATE_WITH_SYSTEM = """<|im_start|>system
{system}<|im_end|>
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
"""

# Optional: get end-of-sequence ID
eos_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

# Generate responses
for i, prompt in enumerate(prompts):
    formatted_prompt = QWEN_CHAT_TEMPLATE_WITH_SYSTEM.format(
        system=system_prompts[i],
        instruction=prompt
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            top_p=0.95,
            temperature=0.7,
            eos_token_id=eos_token_id
        )

    # Slice generated output and clean
    raw_output = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(raw_output, skip_special_tokens=True).split("<|im_end|>")[0].strip()

    print(f"\nPrompt: {prompt}\nResponse: {response}")


/home/kartik/refusal_direction/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
The model is automatically converting to bf16 for faster inference. If you want to disable the automatic precision, please manually add bf16/fp16/fp32=True to "AutoModelForCausalLM.from_pretrained".
Try importing flash-attention for faster inference...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

RuntimeError: cutlassF: no kernel found to launch!